<a href="https://colab.research.google.com/github/PaulF-Analytics/African-Industrial-Paradox-Analysis/blob/main/Industrial_Synchronization_Analysisipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### African Industrial Synchronization Paradox Study (1996–2024)
## A Longitudinal Analysis of Nigeria, South Africa, Kenya, and Rwanda
**Author:** Paul Funsho Wande  
**Role:** Lead Data Analyst
---
**Project Objective:** To analyze the disconnect between wealth accumulation and industrial
output across major African economies using World Bank API data.


In [ ]:
# Setup: Install necessary libraries
!pip install wbgapi --quiet

import pandas as pd
import wbgapi as wb
import matplotlib.pyplot as plt

print("Environment Ready.")

In [ ]:
# 1. Setup
countries = ['NGA', 'ZAF', 'RWA', 'KEN']
years = range(1996, 2025)
indicators_map = {
    'PV.EST': 'political_stability',
    'NV.IND.MANF.ZS': 'manufacturing_output',
    'BX.KLT.DINV.WD.GD.ZS': 'fdi_pct_gdp',
    'FP.CPI.TOTL.ZG': 'inflation',
    'NY.GDP.PCAP.PP.CD': 'gdp_purchasing_power'
}

print("Fetching data... (This may take a moment for 28 years)")
df_raw = wb.data.DataFrame(list(indicators_map.keys()), countries, time=years, labels=True)

# 2. Flatten the index
history_df = df_raw.reset_index()

# 3. Rename columns based on the indicators map
history_df.rename(columns=indicators_map, inplace=True)

# 4. DYNAMIC SEARCH: Find the Year and Country columns no matter what they are called
for col in history_df.columns:
    # If the column name contains 'time' or 'year' (case insensitive)
    if 'time' in col.lower() or 'year' in col.lower():
        history_df.rename(columns={col: 'year'}, inplace=True)
    # If the column contains 'economy' or 'country'
    if 'economy' in col.lower() or 'country' in col.lower():
        history_df.rename(columns={col: 'country'}, inplace=True)

# 5. Clean the 'year' column values
if 'year' in history_df.columns:
    if history_df['year'].dtype == object:
        # Extract numbers (e.g., 'YR1996' -> 1996)
        history_df['year'] = history_df['year'].str.extract(r'(\d+)').astype(int)
else:
    print("Warning: 'year' column not found. Check history_df.columns")

# 6. Preview the result
print("\n--- 28-Year Dataset Success ---")
# Using the country ID usually returned by the index (e.g., NGA)
# If the code below fails, just use print(history_df.head(10))
try:
    print(history_df.sort_values(['country', 'year']).head(10))
except:
    print(history_df.head(10))


In [ ]:
# 1. Deduplicate and clean columns
# We create a clean copy and only take the first instance of any duplicate names
history_df_clean = history_df.loc[:, ~history_df.columns.duplicated()].copy()

# 2. Identify ID and Year columns
# We'll use the column at index 2 (Country Name) and index 1 (Series Name) as IDs
id_vars = [history_df_clean.columns[0], history_df_clean.columns[1]]
year_cols = [c for c in history_df_clean.columns if c.startswith('YR')]

# 3. Melt the data
tidy_df = history_df_clean.melt(id_vars=id_vars, value_vars=year_cols, var_name='Year', value_name='Value')

# 4. Standardize column names
tidy_df.columns = ['country_code', 'indicator_name', 'year', 'value']

# 5. Clean Year (YR1996 -> 1996)
tidy_df['year'] = tidy_df['year'].str.replace('YR', '').astype(int)

# 6. Pivot to get indicators as columns
# This handles the 28-year timeline correctly
final_df = tidy_df.pivot_table(index=['country_code', 'year'],
                               columns='indicator_name',
                               values='value').reset_index()

# 7. Shorten long World Bank names for easier plotting
name_map = {col: col.split(',')[0] for col in final_df.columns if ',' in col}
final_df.rename(columns=name_map, inplace=True)

print("--- 28-Year Historical Data Ready ---")
print(final_df[final_df['country_code'] == 'NGA'].sort_values('year').head(10))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Map the codes directly to the columns we see in your list
mfg_col = 'NV.IND.MANF.ZS'
inf_col = 'FP.CPI.TOTL.ZG'
fdi_col = 'BX.KLT.DINV.WD.GD.ZS'
gdp_col = 'NY.GDP.PCAP.PP.CD'

# 2. Setup Plotting
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12), sharex=True)
sns.set_theme(style="whitegrid")

# Chart A: Manufacturing Output (% of GDP)
# This shows the "Industrial Hollowing" you identified
sns.lineplot(data=final_df, x='year', y=mfg_col, hue='country_code',
             linewidth=3, marker='o', ax=ax1)
ax1.set_title('The Industrial Decline (1996-2024)', fontsize=16, fontweight='bold')
ax1.set_ylabel('Manufacturing (% of GDP)')
ax1.axvline(x=1999, color='red', linestyle='--', alpha=0.5)

# Chart B: GDP per Capita (PPP)
# This shows the "Growth Paradox" - wealth rising while industry falls
sns.lineplot(data=final_df, x='year', y=gdp_col, hue='country_code',
             linewidth=3, marker='s', ax=ax2)
ax2.set_title('Economic Growth (GDP per Capita, PPP)', fontsize=16, fontweight='bold')
ax2.set_ylabel('Current International $')

plt.tight_layout()
plt.show()


In [ ]:
final_df.to_csv('Africa_Structural_History_1996_2024.csv', index=False)
print("File saved! You can now download it from the 'Files' folder on the left.")


In [ ]:
import wbgapi as wb
import pandas as pd

countries = ['NGA', 'ZAF', 'RWA', 'KEN']
years = range(1996, 2025)
proxies = {
    'NV.IND.MANF.ZS': 'manufacturing',
    'NY.GDP.PCAP.PP.CD': 'gdp_per_capita',
    'FP.CPI.TOTL.ZG': 'inflation',
    'IQ.CPA.CORR.XQ': 'corruption_rating',
    'IQ.CPA.PRES.XQ': 'public_mgmt_quality',
    'SP.DYN.LE00.IN': 'life_exp_hdi'
}

print("Fetching and Reshaping 28-Year Dataset...")

# 1. Fetch data (this returns a MultiIndex: economy, series)
raw_df = wb.data.DataFrame(list(proxies.keys()), countries, time=years)

# 2. Reset index to turn 'economy' and 'series' into columns
df = raw_df.reset_index()

# 3. Melt the year columns (YR1996, YR1997...) into a single 'year' column
# id_vars are the columns we keep as is
id_vars = ['economy', 'series']
year_cols = [c for c in df.columns if c.startswith('YR')]

df_long = df.melt(id_vars=id_vars, value_vars=year_cols, var_name='year', value_name='value')

# 4. Clean the Year (YR1996 -> 1996)
df_long['year'] = df_long['year'].str.replace('YR', '').astype(int)

# 5. Pivot so each indicator (series) gets its own column
master_df = df_long.pivot_table(index=['economy', 'year'],
                               columns='series',
                               values='value').reset_index()

# 6. Rename columns using your proxy dictionary
master_df.rename(columns=proxies, inplace=True)
master_df.rename(columns={'economy': 'country'}, inplace=True)

print("\n--- Success! Institutional & Economic Dataset Ready ---")
print(master_df[master_df['country'] == 'NGA'].sort_values('year').head(10))

# Save for your project
master_df.to_csv('Africa_Final_Research_Data.csv', index=False)


In [ ]:
import matplotlib.pyplot as plt

# Filter for Nigeria to see the specific trend
nga_data = master_df[master_df['country'] == 'NGA'].sort_values('year')

fig, ax1 = plt.subplots(figsize=(12, 6))

# Plot Manufacturing on the left axis
color = 'tab:red'
ax1.set_xlabel('Year')
ax1.set_ylabel('Manufacturing (% of GDP)', color=color, fontweight='bold')
ax1.plot(nga_data['year'], nga_data['manufacturing'], color=color, linewidth=3, label='Manufacturing')
ax1.tick_params(axis='y', labelcolor=color)

# Create a second axis for GDP
ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('GDP per Capita (PPP)', color=color, fontweight='bold')
ax2.plot(nga_data['year'], nga_data['gdp_per_capita'], color=color, linewidth=3, linestyle='--', label='GDP per Capita')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Nigeria: The Paradox of Growth without Industry (1996-2024)', fontsize=15)
fig.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Load the data we saved
df = pd.read_csv('Africa_Final_Research_Data.csv')
nga = df[df['country'] == 'NGA'].sort_values('year')

fig, ax1 = plt.subplots(figsize=(12, 7))

# 1. Plot Manufacturing (SDG 9)
color = '#e63946' # Red for industry decline
ax1.set_xlabel('Year', fontsize=12)
ax1.set_ylabel('Manufacturing (% of GDP)', color=color, fontsize=12, fontweight='bold')
ax1.plot(nga['year'], nga['manufacturing'], color=color, linewidth=4, label='Manufacturing')
ax1.tick_params(axis='y', labelcolor=color)

# 2. Plot GDP per Capita (SDG 8)
ax2 = ax1.twinx()
color = '#457b9d' # Blue for wealth growth
ax2.set_ylabel('GDP per Capita (Wealth)', color=color, fontsize=12, fontweight='bold')
ax2.plot(nga['year'], nga['gdp_per_capita'], color=color, linewidth=4, linestyle='--', label='Wealth (GDP)')
ax2.tick_params(axis='y', labelcolor=color)

# 3. Highlight 1999 (Your Birth Year & NGA Transition)
plt.axvline(x=1999, color='black', linestyle=':', alpha=0.5)
plt.text(1999.5, nga['gdp_per_capita'].min(), '1999 Transition', rotation=90, verticalalignment='bottom')

plt.title('The Paradox: Rising Wealth vs. Shrinking Industry (1996-2024)', fontsize=16, pad=20)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Final safety save
master_df.to_csv('Africa_Final_Master_1996_2024.csv', index=False)
print("Final Master File Created. Download it from the files tab now!")


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Pivot data for a heatmap
inflation_wide = master_df.pivot(index="country", columns="year", values="inflation")

plt.figure(figsize=(14, 6))
# Using a 'YlOrRd' (Yellow-Orange-Red) colormap to show "Heat"
sns.heatmap(inflation_wide, cmap="YlOrRd", annot=False, cbar_kws={'label': 'Inflation Rate %'})

plt.title('Inflation Heatmap: 28 Years of Macroeconomic Volatility', fontsize=15)
plt.show()
